# Fit di Fano delle risonanze nella zona della rampa

Legge un CSV (per esempio `punto_NNN_....csv` con `CH1` e `CH2`), prende lo **spettro** (`CANALE`, di default CH2) **nella zona della rampa di CH1** e fa il fit di una o più risonanze con la forma di Fano:

**y = y0 + A · (q + ε)² / (1 + ε²)**, con **ε = 2 (x − x0) / Γ**

- `x0` posizione della risonanza, `Γ` larghezza (FWHM della parte Lorentziana), `q` parametro di asimmetria (per `q → ∞` la risonanza diventa una Lorentziana simmetrica), `A` ampiezza e `y0` livello di fondo. Lontano dalla risonanza il segnale tende a `y0 + A`. Con `BASELINE_PENDENZA = True` si aggiunge un fondo inclinato `s·(x − x0)`.

**Come lavora:**
1. Zona della rampa: l'intervallo tra l'**inizio e la fine esatti** della rampa di CH1, trovati con un **fit della rampa** (plateau basso, salita lineare, plateau alto) che usa anche alcuni punti **prima dell'inizio e dopo la fine** (`MARGINE_RAMPA`), senza i bordi non validi della media mobile. Il primo campione sopra `V_INIZIO_V` serve solo come stima di partenza.
2. (Con `FIT_SU_TUTTA_LA_RAMPA = True` c'è un solo fit su tutta la zona della rampa.) Altrimenti le risonanze vengono **individuate** sulla curva della media mobile (la `moving_average` di `movemin.py`, `np.convolve`) e il fit di ciascuna è fatto sui **dati** (non sulla media mobile) in una finestra di ± `FINESTRA_FATTORE` larghezze intorno al picco, oppure in `FINESTRA_MANUALE` se la scrivi tu.
3. Il fit è robusto ai picchi spuri (`FIT_ROBUSTO`, perdita `soft_l1`) e parte da più valori di `q`, tenendo il migliore, perché il fit di Fano ha più minimi locali. Si impone `A ≥ 0` (la struttura ha il minimo a `y0` e tende a `y0 + A` lontano dalla risonanza): senza questo vincolo `(A, q)` e `(−A·q², −1/q)` descrivono la stessa curva e `q` non sarebbe unico. Prima del fit i dati della finestra vengono ridotti con la mediana a blocchi (`N_PUNTI_FIT` punti).
4. Per ogni fit: parametri con incertezze, `Q = x0 / Γ` e grafico con i residui.

**Asse x in lunghezza d'onda (predefinito):** l'inizio e la fine della rampa trovati con il fit corrispondono all'inizio e alla fine della scansione del laser, da `LAMBDA_0` a `LAMBDA_1` (1520-1570 nm), e la lunghezza d'onda di ogni campione si ottiene per **proporzione lineare**: `lambda = LAMBDA_0 + (campione - inizio) / (fine - inizio) * (LAMBDA_1 - LAMBDA_0)`. Così `x0` e `Γ` sono in nm e `Q = x0 / Γ` ha il significato fisico usuale. Con `ASSE_X = "campioni"` l'asse resta in campioni.

Ordine: cella 1 (scrivi `PERCORSO_CSV`), 2 (visualizza lo spettro e la zona), 3 (individua le risonanze), 4 (fit), 5 (grafici).

In [ ]:
# Cella 1 - impostazioni e funzioni
import os

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import least_squares
from scipy.signal import find_peaks, peak_widths

PERCORSO_CSV = ""          # <-- SCRIVI QUI il file CSV, es. "2026-10-06/run_007/punto_011_x91.818_y146.526.csv" (relativo al notebook) o percorso completo

CANALE = "CH2"             # lo spettro
CANALE_RAMPA = "CH1"       # la rampa che definisce la zona; None = tutto il record
V_INIZIO_V = 0.9           # rampa di CH1: inizio quando CH1 supera questo valore
TOLLERANZA_FINE_V = 0.3    # rampa di CH1: fine quando CH1 e' a meno di questo dal suo massimo

ASSE_X = "lunghezza_onda"  # "lunghezza_onda" | "campioni"
LAMBDA_0, LAMBDA_1 = 1520.0, 1570.0   # lunghezza d'onda (nm) all'inizio e alla fine della rampa di CH1, cioe' della scansione del laser
MARGINE_RAMPA = 0.25       # per il fit della rampa si usano anche punti prima dell'inizio e dopo la fine: questa frazione della durata della rampa per lato

RISONANZA_VERSO = "massimo"   # "massimo" | "minimo": il picco da individuare sulla curva liscia (il fit poi trova da solo la forma)
N_RISONANZE = 1            # quante risonanze fittare (le piu' prominenti); None = tutte quelle trovate
WINDOW_LENGTH = 10000      # media mobile usata solo per individuare le risonanze (come window_length di movemin.py)
SOGLIA_PROMINENZA = 0.1    # una risonanza viene individuata se la sua altezza sulla curva liscia e' almeno questa frazione dell'escursione nella zona
SNR_MIN = 10               # ... e almeno questo multiplo del rumore residuo della curva liscia

FINESTRA_FATTORE = 3.0     # il fit usa i dati entro +-FINESTRA_FATTORE larghezze (FWHM del picco) dal centro
FINESTRA_MANUALE = None    # (x_min, x_max) nelle unita' dell'asse x per scegliere a mano la zona da fittare; None = automatica
FIT_SU_TUTTA_LA_RAMPA = False   # True: UN solo fit di Fano su CH2 in tutta la zona della rampa di CH1 (per una rampa che contiene una sola risonanza)
BASELINE_PENDENZA = False  # True: aggiunge al modello un fondo inclinato s*(x - x0)
FIT_ROBUSTO = True         # True: perdita soft_l1 (resiste ai picchi spuri); False: minimi quadrati
N_PUNTI_FIT = 5000         # i dati della finestra vengono ridotti a circa questo numero di punti (mediana a blocchi) prima del fit


def moving_average(values, window_length):
    """Media mobile con finestra di lunghezza variabile."""   # identica a movemin.py
    if window_length <= 0:
        raise ValueError('window_length deve essere > 0')

    window_length = int(window_length)
    if window_length % 2 == 0:
        window_length += 1

    kernel = np.ones(window_length) / window_length
    return np.convolve(values, kernel, mode='same')


def rampa_ch1(ch1):
    sopra = np.flatnonzero(ch1 > V_INIZIO_V)
    if sopra.size == 0:
        return None
    i0 = int(sopra[0])
    i1 = i0 + int(np.flatnonzero(ch1[i0:] >= ch1.max() - TOLLERANZA_FINE_V)[0])
    return (i0, i1) if i1 > i0 else None


def fit_rampa(ch1, i0, i1):
    """Inizio e fine esatti della rampa di CH1: fit di un plateau basso, una salita lineare e un plateau alto, che usa anche
    punti prima dell'inizio e dopo la fine (la salita, vista dalla soglia V_INIZIO_V, e' gia' cominciata).
    Restituisce (t_inizio, t_fine, v_basso, v_alto), con t in campioni (numeri reali)."""
    lunghezza = i1 - i0
    margine = int(MARGINE_RAMPA * lunghezza)
    inizio = max(0, i0 - margine - int(0.15 * lunghezza))
    fine = min(len(ch1) - 1, i1 + margine)
    dopo = np.flatnonzero(ch1[i1:fine + 1] < ch1.max() - 1.0)       # il plateau alto finisce quando CH1 comincia a scendere
    if dopo.size:
        fine = i1 + int(dopo[0])
    t = np.arange(inizio, fine + 1, dtype=float)
    v = ch1[inizio:fine + 1]
    passo = max(1, len(t) // 20000)
    t, v = t[::passo], v[::passo]

    def modello(p, tt):
        v_b, v_a, t0, durata = p
        return v_b + (v_a - v_b) * np.clip((tt - t0) / durata, 0.0, 1.0)

    p0 = [float(np.median(v[:max(3, len(v) // 10)])), float(ch1.max()), i0 - 0.1 * lunghezza, 1.1 * lunghezza]
    fit = least_squares(lambda p: modello(p, t) - v, p0, loss="soft_l1", f_scale=0.3,
                        bounds=([-np.inf, -np.inf, inizio, 10.0], [np.inf, np.inf, fine, 3.0 * lunghezza]))
    v_b, v_a, t0, durata = fit.x
    return float(t0), float(t0 + durata), float(v_b), float(v_a)


def fano(x, y0, A, x0, gamma, q, s=0.0):
    """Forma di Fano: y0 + s*(x - x0) + A * (q + eps)^2 / (1 + eps^2), eps = 2 (x - x0) / gamma."""
    eps = 2.0 * (x - x0) / gamma
    return y0 + s * (x - x0) + A * (q + eps) ** 2 / (1.0 + eps ** 2)


def riduci_mediana(x, y, n_punti):
    """Riduce (x, y) a circa n_punti con la mediana a blocchi: toglie i picchi spuri e rende il fit veloce."""
    n = max(1, len(x) // int(n_punti))
    k = len(x) // n
    return x[:k * n].reshape(k, n).mean(axis=1), np.median(y[:k * n].reshape(k, n), axis=1)


def fit_fano(x, y, x0_ini, gamma_ini):
    """Fit di una risonanza di Fano su (x, y); parte da piu' valori di q e dal segno di A e tiene il migliore.
    Restituisce un dizionario con i parametri, le incertezze e il modello valutato."""
    scala = float(gamma_ini)
    u = (x - x0_ini) / scala                      # x normalizzata: la risonanza stia vicino a 0 con larghezza ~1
    escursione = float(np.max(y) - np.min(y))
    bordo = max(3, len(y) // 5)
    y0_ini = float(np.median(np.concatenate([y[:bordo], y[-bordo:]])))
    usa_s = bool(BASELINE_PENDENZA)

    def modello(p, uu):
        return fano(uu, p[0], p[1], p[2], p[3], p[4], p[5] if usa_s else 0.0)

    def residui(p):
        return modello(p, u) - y

    limiti_inf = [-np.inf, 0.0, -2.0, 0.05, -50.0] + ([-np.inf] if usa_s else [])      # A >= 0: convenzione che rende q unico
    limiti_sup = [np.inf, 20 * escursione, 2.0, 20.0, 50.0] + ([np.inf] if usa_s else [])
    f_scale = max(1.4826 * float(np.median(np.abs(y - np.median(y)))) * 0.1, escursione * 0.02, 1e-12)
    migliore = None
    livello_fondo = float(np.percentile(y, 5))                                  # il minimo della struttura
    A_ini = max(y0_ini - livello_fondo, 0.1 * escursione)
    for q_ini in (-3.0, -1.0, -0.3, 0.3, 1.0, 3.0):
        p0 = [livello_fondo, A_ini, 0.0, 1.0, q_ini] + ([0.0] if usa_s else [])
        try:
            fit = least_squares(residui, p0, bounds=(limiti_inf, limiti_sup),
                                loss="soft_l1" if FIT_ROBUSTO else "linear", f_scale=f_scale)
        except Exception:
            continue
        if migliore is None or fit.cost < migliore.cost:
            migliore = fit
    if migliore is None:
        raise RuntimeError("Il fit di Fano non e' riuscito")

    p = migliore.x
    n_par = len(p)
    dof = max(1, len(y) - n_par)
    sigma2 = float(np.sum(residui(p) ** 2)) / dof
    cov = np.linalg.pinv(migliore.jac.T @ migliore.jac) * sigma2
    err = np.sqrt(np.clip(np.diag(cov), 0, None))
    x0 = x0_ini + scala * p[2]
    gamma = scala * p[3]
    risultato = {
        "y0": p[0], "A": p[1], "x0": x0, "gamma": gamma, "q": p[4], "s": (p[5] / scala) if usa_s else 0.0,
        "err_y0": err[0], "err_A": err[1], "err_x0": scala * err[2], "err_gamma": scala * err[3], "err_q": err[4],
        "rms": float(np.sqrt(np.mean(residui(p) ** 2))),
        "Q": abs(x0) / gamma,
        "modello": lambda xx: fano(xx, p[0], p[1], x0, gamma, p[4], (p[5] / scala) if usa_s else 0.0),
    }
    risultato["err_Q"] = risultato["Q"] * np.hypot(risultato["err_x0"] / x0 if x0 else 0.0, risultato["err_gamma"] / gamma)
    return risultato

In [ ]:
# Cella 2 - legge lo spettro, trova la zona della rampa e imposta l'asse x
if not PERCORSO_CSV.strip():
    raise ValueError("Scrivi in cella 1 il file CSV in PERCORSO_CSV")
percorso = os.path.abspath(PERCORSO_CSV)
df = pd.read_csv(percorso)
print(f"File: {percorso} | {len(df)} righe, colonne: {list(df.columns)}")
if CANALE not in df.columns:
    raise ValueError(f"Colonna {CANALE!r} non trovata: colonne disponibili {list(df.columns)}")
y = df[CANALE].to_numpy(dtype=float)

mezza = (int(WINDOW_LENGTH) + (int(WINDOW_LENGTH) % 2 == 0)) // 2
a, b = mezza, len(y) - 1 - mezza          # via i bordi dove la media mobile con bordi a zero non e' valida
ch1 = None
inizio_rampa = fine_rampa = None
if CANALE_RAMPA and CANALE_RAMPA in df.columns:
    ch1 = df[CANALE_RAMPA].to_numpy(dtype=float)
    rampa = rampa_ch1(ch1)
    if rampa is None:
        raise RuntimeError(f"Nessuna rampa in {CANALE_RAMPA} (mai sopra {V_INIZIO_V} V)")
    inizio_rampa, fine_rampa, v_basso, v_alto = fit_rampa(ch1, rampa[0], rampa[1])
    print(f"Rampa di {CANALE_RAMPA} (fit con punti prima e dopo): inizio al campione {inizio_rampa:.1f}, fine al campione {fine_rampa:.1f} "
          f"({fine_rampa - inizio_rampa:.0f} campioni), da {v_basso:.3f} V a {v_alto:.3f} V")
    a, b = max(a, int(np.ceil(inizio_rampa))), min(b, int(np.floor(fine_rampa)))
if b <= a:
    raise RuntimeError("Zona della rampa vuota")
print(f"Zona della rampa: campioni {a}..{b} ({b - a + 1} campioni)")

indici = np.arange(len(y), dtype=float)
if ASSE_X == "campioni":
    x_tutto, unita_x = indici, "campioni"
elif ASSE_X == "lunghezza_onda":
    if inizio_rampa is None:
        raise ValueError("ASSE_X = 'lunghezza_onda' richiede la colonna della rampa (CANALE_RAMPA)")
    x_tutto = LAMBDA_0 + (indici - inizio_rampa) / (fine_rampa - inizio_rampa) * (LAMBDA_1 - LAMBDA_0)
    unita_x = "nm"
    print(f"Lunghezza d'onda: {x_tutto[a]:.3f} .. {x_tutto[b]:.3f} nm ({(x_tutto[1] - x_tutto[0]) * 1e3:.4f} pm per campione)")
else:
    raise ValueError("ASSE_X deve essere 'campioni' o 'lunghezza_onda'")

if ch1 is not None:
    durata_rampa = fine_rampa - inizio_rampa
    t_fit = np.arange(max(0, int(inizio_rampa - 0.3 * durata_rampa)), min(len(y), int(fine_rampa + 0.3 * durata_rampa)))
    pr = max(1, len(t_fit) // 10000)
    fig, axr = plt.subplots(figsize=(10, 3))
    axr.plot(t_fit[::pr], ch1[t_fit][::pr], color="tab:red", lw=1, label=CANALE_RAMPA)
    axr.axvline(inizio_rampa, color="k", ls="--", label=f"inizio ({LAMBDA_0:g} nm)")
    axr.axvline(fine_rampa, color="k", ls=":", label=f"fine ({LAMBDA_1:g} nm)")
    axr.set_xlabel("campione")
    axr.set_ylabel(f"{CANALE_RAMPA} (V)")
    axr.set_title("Rampa e suo fit: controlla che inizio e fine siano giusti")
    axr.legend(loc="best")
    plt.tight_layout()
    plt.show()

passo = max(1, len(y) // 20000)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_tutto[::passo], y[::passo], color="gray", lw=0.6)
ax.axvspan(x_tutto[a], x_tutto[b], color="orange", alpha=0.15, label="zona della rampa")
ax.set_xlabel(f"x ({unita_x})")
ax.set_ylabel(CANALE)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# Cella 3 - individua le risonanze sulla curva della media mobile (solo per scegliere le finestre di fit)
liscia = moving_average(y, WINDOW_LENGTH)
zona = liscia[a:b + 1]
segno = 1.0 if RISONANZA_VERSO == "massimo" else -1.0
escursione = zona.max() - zona.min()
d = np.diff(y[a:b + 1])
sigma_liscia = 1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2) / np.sqrt(int(WINDOW_LENGTH))
picchi, prop = find_peaks(segno * zona, prominence=max(SOGLIA_PROMINENZA * escursione, SNR_MIN * sigma_liscia),
                          distance=int(WINDOW_LENGTH))
if len(picchi) == 0:
    raise RuntimeError("Nessuna risonanza individuata sulla curva liscia: abbassa SOGLIA_PROMINENZA o SNR_MIN, o controlla RISONANZA_VERSO")
larghezze, _, sx, dx = peak_widths(segno * zona, picchi, rel_height=0.5,
                                   prominence_data=(prop["prominences"], prop["left_bases"], prop["right_bases"]))
trovate = [{"idx": a + int(p), "fwhm_campioni": float(w), "altezza": float(h)}
           for p, w, h in zip(picchi, larghezze, prop["prominences"])]
trovate.sort(key=lambda r: -r["altezza"])                         # dalla piu' prominente
da_fittare = trovate if N_RISONANZE is None else trovate[:int(N_RISONANZE)]
da_fittare.sort(key=lambda r: r["idx"])                           # in ordine di posizione
if FIT_SU_TUTTA_LA_RAMPA:
    da_fittare = [trovate[0]]                                      # un solo fit, partendo dalla risonanza piu' prominente
print(f"Risonanze individuate: {len(trovate)}; ne fitto {len(da_fittare)}:")
for r in da_fittare:
    print(f"  campione {r['idx']} (x = {x_tutto[r['idx']]:.6g} {unita_x}), larghezza ~ {r['fwhm_campioni']:.0f} campioni, altezza {r['altezza']:.4g}")

In [ ]:
# Cella 4 - fit di Fano di ogni risonanza scelta
risultati_fit = []
for n, r in enumerate(da_fittare, start=1):
    if FIT_SU_TUTTA_LA_RAMPA:
        ia, ib = a, b
    elif FINESTRA_MANUALE is not None:
        sel = (x_tutto >= min(FINESTRA_MANUALE)) & (x_tutto <= max(FINESTRA_MANUALE))
        sel[:a] = False
        sel[b + 1:] = False
        ia, ib = int(np.flatnonzero(sel)[0]), int(np.flatnonzero(sel)[-1])
    else:
        mezzo = FINESTRA_FATTORE * r["fwhm_campioni"]
        ia, ib = max(a, int(r["idx"] - mezzo)), min(b, int(r["idx"] + mezzo))
    xw, yw = x_tutto[ia:ib + 1], y[ia:ib + 1]
    xr, yr = riduci_mediana(xw, yw, N_PUNTI_FIT)

    centro_ini = x_tutto[r["idx"]]
    gamma_ini = abs(x_tutto[min(b, r["idx"] + int(r["fwhm_campioni"]))] - x_tutto[r["idx"]]) or abs(xw[-1] - xw[0]) / 10
    fit = fit_fano(xr, yr, centro_ini, gamma_ini)
    fit.update({"n": n, "ia": ia, "ib": ib, "xw": xw, "yw": yw, "xr": xr, "yr": yr})
    risultati_fit.append(fit)

intestazione = f"{'n':>2} {'x0 (' + unita_x + ')':>16} {'Gamma (' + unita_x + ')':>16} {'q':>9} {'A':>11} {'y0':>11} {'Q = x0/Gamma':>14} {'rms':>10}"
print(intestazione)
for f in risultati_fit:
    print(f"{f['n']:>2} {f['x0']:>16.6g} {f['gamma']:>16.5g} {f['q']:>9.3f} {f['A']:>11.4g} {f['y0']:>11.4g} {f['Q']:>14.5g} {f['rms']:>10.3g}")
print()
for f in risultati_fit:
    print(f"Risonanza {f['n']}: x0 = {f['x0']:.6g} +- {f['err_x0']:.2g} | Gamma = {f['gamma']:.5g} +- {f['err_gamma']:.2g} | "
          f"q = {f['q']:.3f} +- {f['err_q']:.2g} | A = {f['A']:.4g} +- {f['err_A']:.2g} | y0 = {f['y0']:.4g} +- {f['err_y0']:.2g} | Q = {f['Q']:.5g} +- {f['err_Q']:.2g}")
if ASSE_X == "campioni":
    print("(l'asse x e' in campioni: Q = x0/Gamma ha significato fisico solo con ASSE_X = 'lunghezza_onda')")

In [ ]:
# Cella 5 - grafici: spettro con i fit e, per ogni risonanza, dati, fit e residui
passo = max(1, len(y) // 20000)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_tutto[::passo], y[::passo], color="gray", lw=0.6, alpha=0.7, label=f"{CANALE} grezzo")
ax.plot(x_tutto[::passo], liscia[::passo], color="red", lw=1.0, alpha=0.7, label=f"media mobile (N={WINDOW_LENGTH})")
ax.axvspan(x_tutto[a], x_tutto[b], color="orange", alpha=0.1)
for f in risultati_fit:
    ax.plot(f["xw"], f["modello"](f["xw"]), color="blue", lw=2)
    ax.annotate(str(f["n"]), (f["x0"], f["modello"](np.array([f["x0"]]))[0]), textcoords="offset points", xytext=(0, 10), ha="center")
ax.set_xlabel(f"x ({unita_x})")
ax.set_ylabel(CANALE)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

fig, assi = plt.subplots(2, len(risultati_fit), figsize=(5.2 * len(risultati_fit), 5.5), sharex="col",
                         gridspec_kw={"height_ratios": [3, 1]}, squeeze=False)
for k, f in enumerate(risultati_fit):
    a1, a2 = assi[0, k], assi[1, k]
    a1.plot(f["xw"], f["yw"], color="lightgray", lw=0.5, label="dati")
    a1.plot(f["xr"], f["yr"], "k.", ms=2, label="mediana a blocchi")
    xx = np.linspace(f["xw"][0], f["xw"][-1], 2000)
    a1.plot(xx, f["modello"](xx), color="blue", lw=2, label="fit di Fano")
    a1.set_title(f"risonanza {f['n']}: x0={f['x0']:.6g}, Gamma={f['gamma']:.4g}, q={f['q']:.2f}", fontsize=10)
    a1.set_ylabel(CANALE)
    a1.legend(loc="best", fontsize=8)
    a2.plot(f["xr"], f["yr"] - f["modello"](f["xr"]), "k.", ms=2)
    a2.axhline(0, color="blue", lw=1)
    a2.set_ylabel("residui")
    a2.set_xlabel(f"x ({unita_x})")
plt.tight_layout()
plt.show()